In [11]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict,Literal
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [ ]:
load_dotenv()

In [43]:
model=ChatOpenAI(model='gpt-4o-mini')

In [14]:
class sentiment_schema (BaseModel):
    sentiment:Literal['positive','negative']=Field(description='Sentiments of the review')

In [21]:
class diagnosis_schema (BaseModel):
    issue_type : Literal ['UX','performance','bug','support','others']= Field(description='the catagory of the issue mentioned in the review')
    tone :Literal ['angry','frustrated','dissappointed','calm']= Field(description='the emotion of the review')
    urgency : Literal ['low','medium','high']=Field(description='how urgent or crticle the issue mentioned in the review is')

In [22]:
structured_model=model.with_structured_output(sentiment_schema)
structured_model_2=model.with_structured_output(diagnosis_schema)

In [41]:
class review_state(TypedDict):
    review:str
    sentiment: Literal["positive","negative"]
    diagnosis: dict
    response:str

In [17]:
def find_sentiment_python (state:review_state):
    prompt =f'For the following review find the sentiment\n {state['review']}'
    sentiment=structured_model.invoke(prompt).sentiment
    return {'sentiment':sentiment}

In [18]:
def check_sentiment (state:review_state)->Literal['positive_diagnosis','run_diagnosis']:
    if state['sentiment']=='positive':
        return 'positive_diagnosis'
    else :
        return 'run_diagnosis'

In [ ]:
def run_diagnosis_python (state:review_state):
    prompt=f"""diagnose this negative review{state['review']} :\n
    "return issue type, tone , urgency """ 

    response= structured_model_2.invoke(prompt)
# reponse is a pydantic/json obj to convert it in dict use ..model_dump()
    return {'diagnosis': response.model_dump() }

In [19]:
def positive_response_python (state:review_state):
    prompt= f"""write a warm thank you message in response to this review:
    \n\n\"{state['review']}\"\n
    also kindly ask the user to leave feedback on our website."""
    response=model.invoke(prompt)

    return {'response':response}

In [ ]:
def negative_response_python (state:review_state):
    diagnosis= state['diagnosis'] 
    prompt= f"""you are a support assistance\n
     the user had a '{diagnosis['issue_type']}' issue , sounded '{diagnosis['tone']}' , 
     and marked urgency as '{diagnosis['urgency']}'.
     Write an empathatic, helpful resolution message.
     """
    response = model.invoke(prompt).content
    return {'response':response}

In [ ]:
graph=StateGraph(review_state)

graph.add_node('find_sentiment',find_sentiment_python)
graph.add_node('run_diagnosis',run_diagnosis_python)
graph.add_node('positive_diagnosis',positive_response_python)
graph.add_node('negative_response',negative_response_python)



graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment)

graph.add_edge('run_diagnosis','negative_response')
graph.add_edge('negative_response',END)
graph.add_edge('positive_diagnosis',END)

workflow=graph.compile()

In [42]:
initial_state = {'review':'the app gets stuck at athentication screen'}
workflow.invoke(initial_state)

{'review': 'the app gets stuck at athentication screen',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'bug', 'tone': 'frustrated', 'urgency': 'high'},
 'response': "Subject: We're Here to Help with Your Issue\n\nHi [User's Name],\n\nThank you for reaching out and sharing your concern with us. I completely understand how frustrating it can be to encounter a bug, and I want to assure you that we're here to help you resolve this as quickly as possible.\n\nCould you please provide me with a bit more detail about the issue you’re experiencing? Specifically, any error messages you see or the steps leading up to the bug would be incredibly helpful. The more information you can provide, the better we can assist you.\n\nI appreciate your patience and understanding, and I’m committed to helping you get this sorted out swiftly. Please let me know how you’d like to proceed!\n\nBest regards,\n\n[Your Name]  \n[Your Position]  \n[Your Contact Information]  "}